# 02 · Tokenizers, models and `generate()`

This notebook opens up what `pipeline()` hides. You will tokenise text and read what comes back, render a chat template, run a causal LM forward pass down to next-token probabilities, control `generate()` with every common knob, count parameters and memory, look inside at hidden states and attention, and rebuild a sentence embedding by hand.

**Time:** ~30 min · **Runs:** offline on CPU in about a minute · **Needs:** [01 · Hub and pipelines](01_hub_and_pipelines.ipynb); [tokenization and BPE](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) and [decoding](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb) cover the theory

## Why this matters in interviews

- Self-hosting a model means you own the tokenizer, the chat template and the decoding settings. Getting any of them wrong degrades the output quietly, with no error.
- "What are logits / what does temperature do / top-k vs top-p?" are standard questions, and showing the real probabilities from a real model is the best answer.
- Memory arithmetic ("how much GPU memory does a 7B model need?") is asked constantly. You should be able to do it from the parameter count and the dtype.
- Hand-rolled mean pooling is a common live-coding request, and so is "why do you need the attention mask?".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo02` | What is a token, and why does everything in this field get measured in tokens? |
| `fo03` | Why do LLMs use a subword tokenizer instead of treating every word as one token? |
| `tn24` | What is a chat template and why does it matter when self-hosting? |
| `fo22` | What are logits, and what is the softmax doing? |
| `fo14` | What is temperature actually doing, mathematically? |
| `fo15` | What are top-k and top-p, and which should you use? |
| `fo16` | What are max tokens, stop sequences and the repetition penalty? |
| `fo23` | What are logprobs and what can you legitimately use them for? |
| `tr23` | What does the unembedding / LM head do, and what is weight tying? |
| `tr42` | The memory footprint of this 3B model is 12.9 GB. Why, and how would you cut it? |

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, GenerationConfig

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
torch.manual_seed(0)
LM = "HuggingFaceTB/SmolLM2-135M-Instruct"
print("transformers", transformers.__version__)

## 1. Tokenizers: text ↔ integer ids

**In plain English:** a model can't read letters. The tokenizer cuts text into pieces from a fixed vocabulary (whole words, parts of words, single bytes) and swaps each piece for its row number in the model's embedding table. Cost, context length and speed are all counted in these pieces (`fo02`).

`AutoTokenizer.from_pretrained` reads `tokenizer.json` from the repo and returns the right class. Calling it on text gives a dict:

| Key | Meaning |
|---|---|
| `input_ids` | the token ids the model reads |
| `attention_mask` | 1 = real token, 0 = padding the model must ignore |
| `token_type_ids` | (BERT-style models only) sentence A = 0, sentence B = 1 |

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
enc = bert_tok("Tokenizers split unbelievably long words.")
print("keys          :", list(enc.keys()))
print("input_ids     :", enc["input_ids"])
print("tokens        :", enc.tokens())
print("decode        :", bert_tok.decode(enc["input_ids"]))
print("decode (clean):", bert_tok.decode(enc["input_ids"], skip_special_tokens=True))
print("special tokens:", bert_tok.special_tokens_map)
assert enc.tokens()[0] == "[CLS]" and enc.tokens()[-1] == "[SEP]"

BERT's tokenizer (**WordPiece**) lowercases the text, marks word continuations with `##`, and wraps the input in `[CLS] ... [SEP]`. A rare word such as "unbelievably" becomes five pieces, while common words stay whole. That is the point of **subword** tokenization (`fo03`): a vocabulary of about 30 k pieces can spell any word, so nothing is ever "unknown", and common words still cost one token.

Now compare three tokenizers on the same text. GPT-2 and SmolLM2 use **byte-level BPE**, where `Ġ` marks a leading space:

In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
smol_tok = AutoTokenizer.from_pretrained(LM)

samples = {
    "english": "Tokenizers split unbelievably long words.",
    "code":    "def add(a, b):\n    return a + b",
    "numbers": "Order 1234567 costs $89.99",
    "other":   "naïve café, Tōkyō, 東京",
}
print(f"{'':9s} {'bert':>5s} {'gpt2':>5s} {'smol':>5s}   smol pieces")
for name, text in samples.items():
    n = [len(t(text, add_special_tokens=False)["input_ids"]) for t in (bert_tok, gpt2_tok, smol_tok)]
    pieces = smol_tok.convert_ids_to_tokens(smol_tok(text)["input_ids"])
    print(f"{name:9s} {n[0]:5d} {n[1]:5d} {n[2]:5d}   {pieces[:9]}")

for t in (gpt2_tok, smol_tok):      # byte-level BPE is lossless: decode(encode(x)) == x
    for text in samples.values():
        assert t.decode(t(text)["input_ids"]) == text

What to notice:

- The **same text costs a different number of tokens** in each tokenizer. Token counts, prices and context limits only make sense *per tokenizer*, so count with the model's own tokenizer.
- SmolLM2 splits numbers into **single digits** (GPT-2 does not). Tokenizer designers choose this on purpose because it helps arithmetic.
- Non-English text and unusual characters cost several tokens each (byte-level BPE falls back to raw UTF-8 bytes, which is where the garbled-looking pieces come from). The same sentence can cost 2–4× more tokens in some languages.
- Byte-level BPE is **lossless**: the asserts check that decoding the encoding gives back the exact text. BERT's WordPiece lowercases and strips accents, so it cannot round-trip.

### Padding, truncation and batches

A batch must be a rectangle, so shorter sequences get **padded** and the `attention_mask` tells the model which positions are padding. **Truncation** cuts anything longer than `max_length`.

In [ ]:
texts = ["short one", "a much longer sentence than the first"]
batch = bert_tok(texts, padding=True, return_tensors="pt")                # pad to the longest in the batch
print("padding=True\n", batch["input_ids"], "\n", batch["attention_mask"])
batch6 = bert_tok(texts, padding="max_length", truncation=True, max_length=6, return_tensors="pt")
print("max_length=6 ->", bert_tok.convert_ids_to_tokens(batch6["input_ids"][1]))
assert batch["attention_mask"][0].sum() == 4 and batch6["input_ids"].shape == (2, 6)

try:                                  # classic gotcha: GPT-2 was trained without a pad token
    gpt2_tok(texts, padding=True)
except ValueError as e:
    print("\nGPT-2 padding ->", str(e)[:75], "...")
gpt2_tok.pad_token = gpt2_tok.eos_token   # the usual fix; the attention mask keeps it harmless
print("after pad_token = eos_token:", gpt2_tok(texts, padding=True)["attention_mask"])

Two padding rules to remember:

- **Encoders and training:** pad on the right (the default).
- **Decoder-only generation in a batch:** pad on the **left** (`tokenizer.padding_side = "left"`). `generate()` appends new tokens at the end, so with right padding a short prompt would continue *after* its padding. Exercise 3 does this.

### Fast tokenizers and offsets

"Fast" tokenizers (Rust, from the `tokenizers` library, now the default) also return **character offsets** and **word ids**. You need these to map token-level predictions (NER tags, highlighted evidence) back onto the original string.

In [ ]:
text = "Hugging Face opened an office in Paris"
enc = bert_tok(text, return_offsets_mapping=True)
print("is_fast:", bert_tok.is_fast)
for tokn, (s, e), w in zip(enc.tokens(), enc["offset_mapping"], enc.word_ids()):
    print(f"{tokn:10s} chars {s:2d}-{e:2d}  word {w}  -> {text[s:e]!r}")
paris = enc.tokens().index("paris")
s, e = enc["offset_mapping"][paris]
assert text[s:e] == "Paris"          # offsets point into the ORIGINAL (cased) text

## 2. Chat templates

**In plain English:** an instruct model was trained on conversations written in one exact format, with special markers for who is speaking. You must send it that same format. The format is stored with the tokenizer as a Jinja template, and `apply_chat_template` fills it in (`tn24`).

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What is a token?"},
]
prompt = smol_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
assert prompt.endswith("<|im_start|>assistant\n")      # add_generation_prompt opens the assistant turn

In [ ]:
no_system = smol_tok.apply_chat_template([messages[1]], tokenize=False, add_generation_prompt=True)
print(no_system)
assert "You are a helpful AI assistant named SmolLM" in no_system    # the template injects a default system prompt

no_gen = smol_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print("without add_generation_prompt, the prompt ends with:", repr(no_gen[-24:]))

What the template reveals:

- SmolLM2 uses the **ChatML** format: `<|im_start|>role\n ... <|im_end|>`. Llama 3, Mistral, Gemma and Qwen each use different markers. If you hand-build a prompt in the wrong format, generation still runs and quietly gets worse.
- If you don't send a system message, **this template inserts one for you**. That surprises people who think they are sending a bare question.
- `add_generation_prompt=True` adds the opening `<|im_start|>assistant\n`, so the model writes the answer. Without it the model may continue the *user's* turn.
- For fine-tuning, use the same template and mask the loss so it only covers the assistant tokens ([SFT with loss masking](../11_finetuning/02_sft_from_scratch_with_loss_masking.ipynb)).

## 3. The model: parameters, weight tying and memory

`AutoModelForCausalLM` reads `config.json`, builds the architecture (`LlamaForCausalLM` here) and loads the safetensors weights. We ask for `float32`. In transformers v5 the default is `dtype="auto"`, which uses the checkpoint's own dtype (bf16 for this model); in v4 the default was fp32.

In [ ]:
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(LM, dtype=torch.float32).eval()
print(f"loaded {type(model).__name__} in {time.perf_counter() - t0:.1f}s")
cfg = model.config
print(f"{cfg.num_hidden_layers} layers, hidden {cfg.hidden_size}, {cfg.num_attention_heads} query heads / "
      f"{cfg.num_key_value_heads} KV heads (grouped-query attention), MLP {cfg.intermediate_size}, vocab {cfg.vocab_size}")
print(model.model.layers[0])

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

total = count(model)                        # parameters() yields each tensor once, even if shared
emb = model.model.embed_tokens.weight.numel()
layer = model.model.layers[0]
attn, mlp = count(layer.self_attn), count(layer.mlp)
print(f"total parameters         : {total:,}   (model.num_parameters() = {model.num_parameters():,})")
print(f"embedding table          : {emb:,}  ({emb / total:.0%} of the model!)")
print(f"per layer: attention {attn:,} + MLP {mlp:,} (MLP share {mlp / (attn + mlp):.0%})")

tied = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()
print("lm_head shares the embedding matrix (weight tying):", tied)
assert tied and total == model.num_parameters()
assert total == emb + cfg.num_hidden_layers * count(layer) + count(model.model.norm)   # lm_head adds 0

Three points interviewers probe (`tr23`, `tr40`-style):

- **Weight tying:** the LM head that turns the last hidden state into 49,152 logits is *the same matrix* as the input embedding table, used transposed. If you add up every printed layer including `lm_head`, you overcount by 28 M parameters.
- In a small model the **embedding table is a large share** of all parameters. In a 70 B model it is only a few percent.
- Within a layer the **MLP holds most of the weights**, which is why LoRA `target_modules` often include `gate_proj/up_proj/down_proj` as well as the attention projections.

**Memory = parameters × bytes per parameter**, plus the KV cache and activations at inference. Let's check that against the real footprint in two dtypes:

In [ ]:
model_bf16 = AutoModelForCausalLM.from_pretrained(LM, dtype=torch.bfloat16)   # what dtype="auto" would give
fp32_mb, bf16_mb = model.get_memory_footprint() / 1e6, model_bf16.get_memory_footprint() / 1e6
print(f"fp32: {fp32_mb:6.1f} MB  (expected {total * 4 / 1e6:.1f} = params x 4 bytes)")
print(f"bf16: {bf16_mb:6.1f} MB  (expected {total * 2 / 1e6:.1f} = params x 2 bytes)")
assert abs(fp32_mb / bf16_mb - 2) < 0.01
del model_bf16

sizes = {"SmolLM2 135M": 0.135e9, "Llama 3.2 3B": 3.2e9, "Llama 3.1 8B": 8e9, "Llama 3.1 70B": 70e9}
bytes_per = {"fp32": 4, "bf16": 2, "int8": 1, "int4": 0.5}
x = np.arange(len(sizes)); w = 0.2
fig, ax = plt.subplots(figsize=(10, 4))
for i, (dt, b) in enumerate(bytes_per.items()):
    ax.bar(x + (i - 1.5) * w, [p * b / 1e9 for p in sizes.values()], w, label=dt)
ax.axhline(80, color="gray", ls="--", lw=1); ax.text(3.25, 90, "one 80 GB GPU", fontsize=8, ha="right")
ax.set_yscale("log"); ax.set_xticks(x, list(sizes)); ax.set_ylabel("weights only (GB, log scale)")
ax.set_title("Weight memory = parameters x bytes per parameter"); ax.legend(ncol=4, fontsize=8)
plt.show()

This is the `tr42` answer: a 3 B model loaded in fp32 needs about 12.8 GB, bf16 halves that at no quality cost relative to the released weights, and int8 or int4 halve it again ([06 · Quantization](06_quantization_with_bitsandbytes.ipynb)). A 70 B model in bf16 (140 GB) does not fit on one 80 GB GPU; in int4 (35 GB) it does. Remember these are *weights only*: serving adds the KV cache, and training adds gradients and optimizer state (roughly 16 bytes per parameter for full fine-tuning with Adam).

bf16 has the same exponent range as fp32 with fewer mantissa bits, which is why it replaced fp16 for training. On CPU, bf16 saves memory but is often *slower*, because many laptop CPUs have no native bf16 maths. That is why these notebooks run the model in fp32.

## 4. Forward pass → logits → next-token probabilities

**In plain English:** for every position in the input, the model outputs one score (a **logit**) for every token in the vocabulary. Softmax turns the scores at the last position into probabilities for the next token (`fo22`).

In [ ]:
enc = smol_tok("The capital of France is", return_tensors="pt")
with torch.no_grad():
    out = model(**enc)
print("input_ids :", tuple(enc["input_ids"].shape), "= [batch, tokens]")
print("logits    :", tuple(out.logits.shape), "= [batch, tokens, vocab]")

next_logits = out.logits[0, -1]                  # only the LAST position predicts the next token
probs = next_logits.softmax(-1)
top = probs.topk(10)
top_tokens = [repr(smol_tok.decode(int(i))) for i in top.indices]
for t, p in zip(top_tokens, top.values):
    print(f"{t:12s} {p:.3f}")
assert abs(probs.sum().item() - 1) < 1e-4

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(top_tokens, top.values.numpy(), color="#4c6ef5")
ax.set_ylabel("probability"); ax.set_title("SmolLM2: next token after 'The capital of France is' (top 10)")
plt.xticks(rotation=30); plt.show()

"Paris" is the top choice, but it gets less than half the probability. "the", "located" and "called" all lead to reasonable sentences too. A language model scores *continuations*, not facts. The logits are raw scores (unbounded, and shifting them all by a constant changes nothing); only after softmax do they become probabilities.

### Temperature, top-k and top-p, on the real distribution

- **Temperature** divides the logits before the softmax: `softmax(z / T)`. T < 1 sharpens the distribution, T > 1 flattens it, and T → 0 becomes greedy (`fo14`).
- **Top-k** keeps the k most likely tokens. **Top-p** (nucleus) keeps the smallest set whose probabilities add up to at least p, so the set shrinks when the model is confident and grows when it isn't (`fo15`).

In [ ]:
temps = [0.5, 1.0, 1.5]
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, T in enumerate(temps):
    pT = (next_logits / T).softmax(-1)
    ax.bar(np.arange(10) + (i - 1) * 0.27, pT[top.indices].numpy(), 0.27, label=f"T = {T}")
ax.set_xticks(np.arange(10), top_tokens, rotation=30)
ax.set_ylabel("probability"); ax.set_title("Temperature reshapes the same logits"); ax.legend()
plt.show()

def nucleus_size(logits, p):
    sp = logits.softmax(-1).sort(descending=True).values
    return int((sp.cumsum(-1) < p).sum()) + 1          # smallest prefix reaching mass p

for T in temps:
    pT = (next_logits / T).softmax(-1)
    print(f"T={T}: P(' Paris') = {pT[top.indices[0]]:.3f}; top-p=0.9 keeps {nucleus_size(next_logits / T, 0.9):5d} tokens")
assert nucleus_size(next_logits / 0.5, 0.9) < nucleus_size(next_logits, 0.9) < nucleus_size(next_logits / 1.5, 0.9)

At T = 0.5, top-p = 0.9 keeps only a handful of tokens. At T = 1.5 it keeps thousands, most of them junk. This is why high temperature produces nonsense, and why top-p is the safer default: the cut-off adapts to how confident the model is.

## 5. `model.generate()` and its knobs

`generate()` runs the loop: forward pass → choose a token → append it → repeat, using the KV cache so earlier positions aren't recomputed. The main parameters:

| Parameter | Effect | Typical |
|---|---|---|
| `max_new_tokens` | hard cap on generated tokens; it cuts, it doesn't summarise | always set it |
| `do_sample` | `False` = greedy (argmax each step); `True` = sample | `False` for extraction/tests |
| `temperature`, `top_k`, `top_p` | shape and truncate the distribution (only when `do_sample=True`) | 0.7, 50, 0.9 for chat |
| `repetition_penalty` | > 1 lowers the logits of tokens already used | 1.0–1.2; hurts code and JSON |
| `num_return_sequences` | several samples from one prompt, in one batch | best-of-n, self-consistency |
| `eos_token_id`, `stop_strings` | where to stop | from `generation_config` |

In [ ]:
chat = smol_tok.apply_chat_template([{"role": "user", "content": "Describe the sea in one sentence."}],
                                    add_generation_prompt=True, return_tensors="pt")   # v5: a dict (BatchEncoding)
n_prompt = chat["input_ids"].shape[1]

def new_text(ids):
    return smol_tok.decode(ids[n_prompt:], skip_special_tokens=True)

t0 = time.perf_counter()
greedy = model.generate(**chat, max_new_tokens=25, do_sample=False)
print(f"greedy ({time.perf_counter() - t0:.1f}s):", new_text(greedy[0]))

with torch.no_grad():                                 # greedy's first token must be the argmax of one forward pass
    first = model(**chat).logits[0, -1].argmax()
assert greedy[0, n_prompt] == first

In [ ]:
torch.manual_seed(0)
samples = model.generate(**chat, max_new_tokens=20, do_sample=True, temperature=0.9, top_p=0.9,
                         num_return_sequences=3)
for i, s in enumerate(samples):
    print(f"sample {i}:", new_text(s))
print("output shape:", tuple(samples.shape), "= [num_return_sequences, prompt + new tokens]")
assert len({new_text(s) for s in samples}) > 1      # sampling gives different continuations

Greedy decoding gives the same text on every run, which is what you want for tests and extraction. Sampling gives variety, which is useful for creative work, for best-of-n and for self-consistency voting. The seed makes the samples repeatable here. In production, batching and GPU kernels can still change the result, so "temperature 0" means greedy, not guaranteed reproducible.

`GenerationConfig` bundles the settings into one object that you can save with the model (`generation_config.json`) and version like code:

In [ ]:
print("defaults shipped with the model:", model.generation_config.to_diff_dict())

creative = GenerationConfig(max_new_tokens=20, do_sample=True, temperature=0.8, top_k=50, top_p=0.95,
                            repetition_penalty=1.3, pad_token_id=smol_tok.pad_token_id,
                            eos_token_id=smol_tok.eos_token_id)
torch.manual_seed(0)
r = model.generate(**chat, generation_config=creative, return_dict_in_generate=True, output_scores=True)
print("with repetition_penalty=1.3:", new_text(r.sequences[0]))

logprobs = model.compute_transition_scores(r.sequences, r.scores, normalize_logits=True)[0]
gen_ids = r.sequences[0, n_prompt:]
print("\nper-token logprobs (the numbers the logprobs API returns):")
for tid, lp in list(zip(gen_ids, logprobs))[:8]:
    print(f"  {smol_tok.decode(int(tid))!r:14s} logprob {lp:6.2f}  p = {lp.exp():.2f}")
assert torch.all(logprobs <= 0)

`output_scores=True` gives the processed scores at each step, and `compute_transition_scores` turns them into the log-probability of each token actually chosen. That is what an API's `logprobs` field contains (`fo23`). It is useful for cheap classification (compare P("yes") with P("no")) and for flagging low-confidence spans. It is not a hallucination detector: a model can be confident and wrong.

## 6. Looking inside: hidden states and attention

Pass `output_hidden_states=True` and `output_attentions=True` to get everything out of the forward pass. By default v5 uses the fused `sdpa` attention kernel, which never builds the attention matrix, so it returns an empty tuple. Switch to `"eager"` attention to see the weights (the logits are the same either way).

In [ ]:
model.set_attn_implementation("eager")
enc = smol_tok("The cat sat on the mat because it was tired", return_tensors="pt")
with torch.no_grad():
    out = model(**enc, output_hidden_states=True, output_attentions=True)
print("hidden_states:", len(out.hidden_states), "tensors (embeddings + one per layer), each", tuple(out.hidden_states[0].shape))
print("attentions   :", len(out.attentions), "tensors (one per layer), each", tuple(out.attentions[0].shape), "= [batch, heads, query, key]")

A = out.attentions[-1][0].mean(0)           # last layer, averaged over the 9 heads
assert torch.allclose(A.sum(-1), torch.ones(A.shape[0]), atol=1e-4)    # each row is a softmax
assert torch.all(A.triu(1) < 1e-6)                                      # causal mask: no attending to the future

toks = [smol_tok.decode(int(t)) for t in enc["input_ids"][0]]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(A.numpy(), cmap="Blues")
ax.set_xticks(range(len(toks)), toks, rotation=60); ax.set_yticks(range(len(toks)), toks)
ax.set_xlabel("attends to (key)"); ax.set_ylabel("token (query)")
ax.set_title("SmolLM2 last layer, mean over heads"); plt.colorbar(im, fraction=0.046); plt.show()
model.set_attn_implementation("sdpa")

How to read the heatmap:

- It is **lower-triangular**, because of the causal mask (`tr06`). Each token attends only to itself and to earlier tokens, and the assert checks it.
- Each row sums to 1: every row is a softmax over the visible positions.
- A lot of the weight goes to the **first token** of the sequence. This "attention sink" is normal in trained LLMs: heads with nothing useful to look at park their weight there (`tr27`).

`hidden_states[i]` is the residual stream after layer *i*. These vectors are what probing, steering and embedding extraction work with.

## 7. Embeddings from `AutoModel` + mean pooling

An encoder like all-MiniLM outputs one vector per token (`last_hidden_state`). A sentence embedding is the **average over the real tokens**. The attention mask must drive the average, otherwise padding tokens get averaged in. sentence-transformers does exactly this (mean pooling, then L2 normalisation), and the asserts check we match it.

In [ ]:
from sentence_transformers import SentenceTransformer

EMB = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok, encoder = AutoTokenizer.from_pretrained(EMB), AutoModel.from_pretrained(EMB).eval()
sentences = ["The cat sits on the mat.", "A feline is resting on a rug.", "Stock markets fell sharply today.", "Hi"]

batch = emb_tok(sentences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    tokens_out = encoder(**batch).last_hidden_state                  # [batch, tokens, 384]
mask = batch["attention_mask"].unsqueeze(-1).float()                 # [batch, tokens, 1]
mean_pooled = (tokens_out * mask).sum(1) / mask.sum(1)               # average over REAL tokens only
manual = torch.nn.functional.normalize(mean_pooled, dim=-1)

st_emb = SentenceTransformer(EMB, device="cpu").encode(sentences, normalize_embeddings=True, convert_to_tensor=True)
print("manual vs sentence-transformers, max |diff|:", f"{(manual - st_emb).abs().max():.2e}")
assert torch.allclose(manual, st_emb, atol=1e-5)

naive = torch.nn.functional.normalize(tokens_out.mean(1), dim=-1)   # forgets the mask
drift = 1 - (naive * st_emb).sum(-1)
print("1 - cos(no-mask mean, correct) per sentence:", np.abs(drift.numpy()).round(3))
print("cosine(cat, feline) =", f"{(manual[0] @ manual[1]):.3f}", "| cosine(cat, stocks) =", f"{(manual[0] @ manual[2]):.3f}")
assert drift[0] > drift[1] - 1e-6 and drift[3] == drift.max()      # the most-padded sentence ("Hi") drifts most

The longest sentence has no padding, so forgetting the mask changes nothing for it (a drift of 0). "Hi" is mostly padding, so its "embedding" moves a long way. That is the bug the attention mask prevents, and a favourite follow-up question.

## Try it yourself

**1. Token budget.** A prompt template is `"Summarise this ticket:\n" + ticket`. Using SmolLM2's tokenizer, find how many tokens are left for the ticket if the model's context is 8192 tokens and you reserve 512 for the answer. Then truncate a very long ticket to fit, using `truncation=True, max_length=...`.

In [ ]:
# Solution — try it yourself first, then run this
CONTEXT, RESERVE = 8192, 512
prefix = "Summarise this ticket:\n"
budget = CONTEXT - RESERVE - len(smol_tok(prefix)["input_ids"])
ticket = "The printer on floor 3 keeps jamming. " * 2000
ids = smol_tok(ticket, truncation=True, max_length=budget)["input_ids"]
print(f"ticket budget: {budget} tokens; the full ticket has {len(smol_tok(ticket)['input_ids'])}; kept {len(ids)}")
assert len(ids) == budget

**2. Top-k vs top-p by hand.** For the prompt "I like to eat", compute how many tokens top-p = 0.9 keeps, and compare with a fixed top-k = 50. Is the model more or less confident here than after "The capital of France is"?

In [ ]:
# Solution — try it yourself first, then run this
with torch.no_grad():
    lg = model(**smol_tok("I like to eat", return_tensors="pt")).logits[0, -1]
k_top = lg.softmax(-1).topk(50).values.sum()
print(f"top-p=0.9 keeps {nucleus_size(lg, 0.9)} tokens; top-k=50 covers {k_top:.1%} of the probability mass")
print(f"after 'The capital of France is', top-p=0.9 kept {nucleus_size(next_logits, 0.9)} tokens")
# The open-ended prompt spreads its probability over many more tokens: top-p grows with it, top-k cannot.

**3. Batched generation needs left padding.** Generate 8 greedy tokens for two prompts of different lengths in one batch, with `padding_side="left"`, and check that the first prompt's continuation matches generating it on its own.

In [ ]:
# Solution — try it yourself first, then run this
prompts = ["The capital of France is", "Once upon a time, in a small village by the sea, there lived"]
smol_tok.padding_side = "left"
b = smol_tok(prompts, padding=True, return_tensors="pt")
batched = model.generate(**b, max_new_tokens=8, do_sample=False)
alone = model.generate(**smol_tok(prompts[0], return_tensors="pt"), max_new_tokens=8, do_sample=False)
width = b["input_ids"].shape[1]
for p, row in zip(prompts, batched):
    print(repr(p), "->", repr(smol_tok.decode(row[width:], skip_special_tokens=True)))
assert smol_tok.decode(batched[0, width:]) == smol_tok.decode(alone[0, -8:])
smol_tok.padding_side = "right"

## Say it in an interview

- **Tokenizer:** "It maps text to ids from a fixed subword vocabulary: WordPiece for BERT, byte-level BPE for GPT-style models. It returns `input_ids` plus an `attention_mask` for padding. Token counts are tokenizer-specific, so I count with the model's own tokenizer." Non-English text and code cost more tokens.
- **Chat template:** "Instruct models are trained on one exact conversation format. `apply_chat_template(..., add_generation_prompt=True)` renders it. A wrong format gives no error, just worse output. Some templates inject a default system prompt."
- **Logits → text:** logits `[batch, seq, vocab]`; only the last position matters for the next token; softmax(z/T); top-k/top-p truncate; greedy is argmax. `max_new_tokens` is a guillotine. `repetition_penalty` breaks JSON and code.
- **Numbers:** memory = params × bytes (fp32 4, bf16 2, int8 1, int4 0.5), weights only. 7 B in bf16 ≈ 14 GB; 70 B in int4 ≈ 35 GB. Full fine-tuning with Adam needs about 16 bytes per parameter.
- **Traps:** the default dtype changed to `"auto"` in transformers v5 (it was fp32 in v4). GPT-2 has no pad token. Decoder batches must pad on the left. `output_attentions` needs eager attention. Mean pooling must use the attention mask. Tied embeddings mean you shouldn't double-count `lm_head`.
- **Follow-ups:** "why is decoding slow?" (one forward pass per token and memory-bound: see [KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)), "how do you get deterministic output?" (greedy plus a fixed seed and fixed batch, and even then GPUs can differ).

## Next

- [03 · The datasets library](03_datasets_library.ipynb) · [04 · Sentence-transformers](04_sentence_transformers_embeddings_and_cross_encoders.ipynb) · [06 · Quantization with bitsandbytes](06_quantization_with_bitsandbytes.ipynb)
- From scratch: [attention](../07_genai_foundations/03_attention_from_scratch.ipynb), [a tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb), [decoding strategies](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb), [KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)
- Fine-tuning with the chat template: [SFT with loss masking](../11_finetuning/02_sft_from_scratch_with_loss_masking.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [interview bank](../../ai_interview_prep/index.html)